# Multilingual — maths and knowledge in many languages (local)

Does the model work as well in Swahili or Hindi as in English? Two benchmarks answer that:

- **MGSM** — 250 grade-school maths word problems, professionally translated into 11 languages (Bengali, German, English, Spanish, French, Japanese, Russian, Swahili, Telugu, Thai, Chinese). Tests *reasoning* in each language.
- **Global-MMLU Lite** — 400 multiple-choice knowledge questions in 15 languages (adds Arabic, Hindi, Indonesian, Italian, Korean, Portuguese, Yoruba). Half are *culturally sensitive* (they depend on local knowledge) and half *culturally agnostic*, so you can see whether the model's knowledge is world-wide or Western-centric.

The score is the average of the two. This notebook runs **locally**, straight against the model's chat endpoint — no EvalHub and no OpenShift. All settings are in the configuration cell below.

**How it works:** [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) (`lm_eval`) is run through `uvx`, once per benchmark. The stock MGSM task cuts a chat model's working off at the first line break, so both tasks are defined in `lm_eval_tasks/` instead.

**Requires:** [`uv`](https://docs.astral.sh/uv/) (for `uvx`) and network access to the model endpoint and Hugging Face (for the datasets).

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, os, sys, shutil, subprocess
from pathlib import Path

# eval_env.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_env.py").is_file())))
from eval_env import setup

# Model endpoint and optional proxy, from .env (or prompts); see eval_env.py
MODEL_URL, MODEL_NAME, MODEL_API_KEY = setup()


# --- Benchmark ---
MGSM_TASK = "mgsm_chat"                          # custom lm-eval tasks, defined in lm_eval_tasks/
GMMLU_TASK = "global_mmlu_lite_generative"
LM_EVAL_VERSION = "0.4.12"  # pinned so results are repeatable
LM_EVAL_EXTRAS = "api"  # pip extras lm-eval needs for these tasks
PYPI_INDEX = "https://pypi.org/simple"  # package index used to install lm-eval, instead of any index set in your uv config
NUM_CONCURRENT = 8          # parallel requests to the model
THRESHOLD = 0.6  # minimum average score to count as PASS
MGSM_MAX_GEN_TOKS = 2048    # max tokens per maths answer (step-by-step working)
GMMLU_NUM_FEWSHOT = 5       # worked examples sent as prior chat turns; teaches the model to reply with just a letter
GMMLU_SYSTEM_INSTRUCTION = "Reply with only the letter of the correct answer (A, B, C, or D)."  # exact_match needs a bare letter
GMMLU_MAX_GEN_TOKS = 32     # those replies are a single letter
LIMIT = None                # per language, e.g. 5 for a quick smoke test; None runs everything
TASKS_DIR = next(d for d in (Path("lm_eval_tasks"), Path("notebooks/lm_eval_tasks")) if d.is_dir()).resolve()  # custom task definitions

## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run the benchmarks

Runs MGSM, then Global-MMLU Lite. Completed requests are cached on disk, so if a run dies partway, re-running this cell resumes where it left off.

In [ ]:
def lm_eval(task, *extra):
    """Run one lm-evaluation-harness task and return the folder holding its results and samples.

    Completed requests are cached, so if the run dies partway, re-running resumes where it stopped;
    the cache is deleted after a successful run so the next run gets fresh answers from the model."""
    output = Path(f".eval-notebook-{task}")
    cache = Path(f".eval-notebook-{task}-cache") / "requests"
    cache.parent.mkdir(exist_ok=True)
    shutil.rmtree(output, ignore_errors=True)
    cmd = ["uvx", "--default-index", PYPI_INDEX, "--python", "3.12", "--from", f"lm-eval[{LM_EVAL_EXTRAS}]=={LM_EVAL_VERSION}",
           "lm_eval", "--model", "local-chat-completions",
           "--model_args", f"model={MODEL_NAME},base_url={MODEL_URL.rstrip('/')}/chat/completions,"
                           f"num_concurrent={NUM_CONCURRENT},max_retries=5,tokenized_requests=False",
           "--tasks", task, "--apply_chat_template", "--log_samples",
           "--output_path", str(output), "--use_cache", str(cache), *extra]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    # Clear uv index overrides inherited from the environment so lm-eval installs from PYPI_INDEX.
    uv_env = {k: "" for k in ("UV_INDEX", "UV_INDEX_URL", "UV_EXTRA_INDEX_URL", "UV_DEFAULT_INDEX")}
    run(cmd, env={**uv_env, "OPENAI_API_KEY": MODEL_API_KEY or "not-required"})
    shutil.rmtree(cache.parent, ignore_errors=True)
    return output

MGSM_OUTPUT = lm_eval(MGSM_TASK, "--include_path", str(TASKS_DIR), "--gen_kwargs", f"max_gen_toks={MGSM_MAX_GEN_TOKS}")
GMMLU_OUTPUT = lm_eval(GMMLU_TASK, "--include_path", str(TASKS_DIR), "--num_fewshot", str(GMMLU_NUM_FEWSHOT),
                       "--system_instruction", GMMLU_SYSTEM_INSTRUCTION, "--gen_kwargs", f"max_gen_toks={GMMLU_MAX_GEN_TOKS}")

## 3. Score and pass/fail

Each benchmark's score is `exact_match` over all its languages; the headline is the average of the two. Per-question answers are in the `.eval-notebook-<task>` folders (`samples_*.jsonl`).

In [ ]:
def headline(output, task):
    results = json.loads(next(output.rglob("results_*.json")).read_text())["results"]
    return next(v for k, v in results[task].items() if k.startswith("exact_match,"))

mgsm, gmmlu = headline(MGSM_OUTPUT, MGSM_TASK), headline(GMMLU_OUTPUT, GMMLU_TASK)
score = (mgsm + gmmlu) / 2
print(f"MGSM {mgsm:.3f}   Global-MMLU Lite {gmmlu:.3f}")
print("PASS" if score >= THRESHOLD else "FAIL", f"- average {score:.3f} vs threshold {THRESHOLD}")

## 4. Results dashboard

Accuracy for every language with 95% confidence intervals, how far each language falls behind English, culturally sensitive vs. culturally agnostic knowledge, and a check for answers that could not be read (for example, numbers written in a local script).

In [ ]:
import sys
from pathlib import Path
# eval_report.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_report.py").is_file())))
from eval_report import report_multilingual

report_multilingual(MGSM_OUTPUT, GMMLU_OUTPUT, THRESHOLD)